## Inferring the laterality from CFI_Keypoints model output

In [ ]:
from eyened_orm import Database, AttributeValue, Modality, AttributeDefinition, ImageInstance, Series, Study, Patient, Project, AttributesModel
database = Database()

projectname = "Your project name"
## Set to True to update the database value 
update_database = False

In [38]:
from eyened_orm import Laterality

def infer_laterality(cfi_keypoints):
    x_fovea = cfi_keypoints["fovea_xy"][0]
    x_disc = cfi_keypoints["disc_edge_xy"][0]
    if x_fovea < x_disc:
        return Laterality.R
    else:
        return Laterality.L
    
    
    

In [ ]:
from tqdm import tqdm

with database.get_session() as session:
    attribute_definition = session.query(AttributeDefinition).filter(AttributeDefinition.AttributeName == "CFI_Keypoints").first()
    results = session.query(ImageInstance, AttributeValue.ValueJSON)\
        .join(AttributeValue, ImageInstance.ImageInstanceID == AttributeValue.ImageInstanceID) \
        .join(AttributesModel, AttributeValue.ModelID == AttributesModel.ModelID).filter(AttributesModel.OutputAttributes.contains(attribute_definition))\
        .join(Series, ImageInstance.SeriesID==Series.SeriesID).join(Study).join(Patient).join(Project)\
        .filter(Project.ProjectName == projectname)\
        .all()
    print(len(results))
    for r in tqdm(results):
        laterality = infer_laterality(r[1])
        if r[0].Laterality is None:
            print("Laterality was not set for image ", r[0].ImageInstanceID, " found laterality: ", laterality)
        elif r[0].Laterality != laterality:
            print("Different laterality inferred for image ", r[0].ImageInstanceID)
        if update_database:
            r[0].Laterality = laterality
            session.commit()
        
    





